# Pipeline 04Ga: global template baseline (deterministic, non-LLM)

Two deterministic baselines built from the global importances + shape/dependence curves,
no API call ($0):
1. **Whole-model** summary per model — `[MODEL] / [DRIVERS] / [RECOMMENDATION]` →
   `results/04Ga/global_{model}.json`.
2. **Per-feature** description — `[EFFECT] / [IMPORTANCE] / [RECOMMENDATION]` for each
   feature → `results/global/template_{model}_{feature}.json`. This is the Non-LLM
   comparison floor for the G2a LLM pipelines (`04Gb/04Gc/04Gd`).

Curve facts come from `utils.describe_curve` — the single source of truth shared with the
G3 ground truth. Split out of `04La` (02.07.); `04La` remains the local baseline.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    RESULTS_DIR, EXPLANATIONS_DIR,
    describe_curve, feature_importance_map, list_global_features,
    build_global_record, run_resumable_global_generation,
)

LOSS_KEY   = 'poisson_log'
XAI_MODELS = ['xgb', 'ebm']

OUT_DIR    = RESULTS_DIR / '04Ga'      # whole-model summary
GLOBAL_DIR = RESULTS_DIR / 'global'    # per-feature baseline (G3 comparison set)
OUT_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Models:      {XAI_MODELS}')
print(f'Whole-model: {OUT_DIR}')
print(f'Per-feature: {GLOBAL_DIR}')

Models:      ['xgb', 'ebm']
Whole-model: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/04Ga
Per-feature: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/global


In [2]:
# Readable feature names for the descriptions. (Value denormalisation and curve
# derivation now live in utils: readable_feature_value / describe_curve.)
FEATURE_NAMES = {
    'hr':         'hour',
    'temp':       'temperature',
    'hum':        'humidity',
    'windspeed':  'wind speed',
    'yr':         'year',
    'mnth':       'month',
    'weekday':    'weekday',
    'weathersit': 'weather',
    'holiday':    'holiday status',
}
print('Feature names loaded.')

Feature names loaded.


In [3]:
# --- Global whole-model baseline: [MODEL] / [DRIVERS] / [RECOMMENDATION] ---
# The local [PREDICTION] section is repurposed into a global [MODEL] overview
# (design decision, see planning/Revision_30062026_Umsetzungsplan.md). Deterministic.
# Direction/peak come from utils.describe_curve (shared with the per-feature baseline
# below and the G3 ground truth).
import math


def generate_global_template(global_data: dict, xai: str) -> str:
    """Deterministic whole-model description of an EBM/XGB from the global JSON.
    Structure: [MODEL] / [DRIVERS] / [RECOMMENDATION]."""
    task     = global_data['task']['description']
    baseline = math.exp(global_data['base_value'])   # log space -> cnt space
    metrics  = global_data.get('metrics', {})
    top3     = global_data['global_importance'][:3]

    fit_keys = ('rmse', 'mae', 'r2', 'poisson_deviance')
    metric_str = ', '.join(f'{k}={metrics[k]:.3f}' for k in fit_keys if k in metrics) or 'n/a'
    model = (
        f'[MODEL] The model predicts {task[0].lower() + task[1:]}. On average it expects '
        f'about {baseline:.0f} rentals per hour (baseline). Model fit: {metric_str}.'
    )

    lines = []
    for item in top3:
        feat = item['feature']
        d = describe_curve(xai, feat, explanations_dir=EXPLANATIONS_DIR)
        lines.append(
            f"* {FEATURE_NAMES.get(feat, feat)} (rank {item['rank']}, "
            f"importance {item['importance']:.3f}): {d['direction']} effect, "
            f"contribution highest around {d['peak_label']}."
        )
    drivers = (
        '[DRIVERS] The features with the largest overall influence on demand are:\n'
        + '\n'.join(lines)
    )

    top      = top3[0]
    top_name = FEATURE_NAMES.get(top['feature'], top['feature'])
    top_peak = describe_curve(xai, top['feature'], explanations_dir=EXPLANATIONS_DIR)['peak_label']
    recommendation = (
        f'[RECOMMENDATION] Plan capacity around the dominant driver ({top_name}): '
        f'provide the most bikes and staff at its high-demand level (around {top_peak}) '
        f'and scale resources down otherwise.'
    )
    return '\n\n'.join([model, drivers, recommendation])


# Smoke test
_gdata = json.loads((EXPLANATIONS_DIR / f'global_ebm_{LOSS_KEY}.json').read_text())
print(generate_global_template(_gdata, 'ebm'))

[MODEL] The model predicts number of rented bikes per hour (sum of casual and registered users). On average it expects about 101 rentals per hour (baseline). Model fit: rmse=48.203, mae=28.198, r2=0.927, poisson_deviance=10.815.

[DRIVERS] The features with the largest overall influence on demand are:
* hour (rank 1, importance 0.889): mixed effect, contribution highest around 17:00.
* temperature (rank 2, importance 0.253): mixed effect, contribution highest around ~31.2 C.
* year (rank 3, importance 0.217): rising effect, contribution highest around 2012.

[RECOMMENDATION] Plan capacity around the dominant driver (hour): provide the most bikes and staff at its high-demand level (around 17:00) and scale resources down otherwise.


In [4]:
# Run the global baseline for both models. Deterministic, no API, $0.
for xai in XAI_MODELS:
    gdata = json.loads((EXPLANATIONS_DIR / f'global_{xai}_{LOSS_KEY}.json').read_text())

    t0          = time.perf_counter()
    explanation = generate_global_template(gdata, xai)
    elapsed     = round(time.perf_counter() - t0, 6)

    record = {
        'explanation':  explanation,
        'elapsed_s':    elapsed,
        'usage':        {'input_tokens': 0, 'output_tokens': 0, 'cache_read_input_tokens': 0},
        'n_tool_calls': 0,
        'scope':        'global',
        'model':        xai,
    }
    out_file = OUT_DIR / f'global_{xai}.json'
    out_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
    print(f'  {xai.upper()} global: {len(explanation.split())} words -> {out_file.name}')

print(f'\nDone. Global baselines saved in {OUT_DIR} (no API call, $0).')

  XGB global: 105 words -> global_xgb.json
  EBM global: 105 words -> global_ebm.json

Done. Global baselines saved in /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/04Ga (no API call, $0).


## Per-feature baseline (deterministic, non-LLM — the G3 comparison floor)

For each feature, a rule-based `[EFFECT] / [IMPORTANCE] / [RECOMMENDATION]` description
built from `utils.describe_curve` + the global importances (the same fields the G3 ground
truth is derived from). Output structure matches the G2a LLM prompt, so one rubric scores
both. Output: `results/global/template_{model}_{feature}.json`.

In [5]:
_DIRECTION = {
    'rising':  'increasing — higher values raise demand',
    'falling': 'decreasing — higher values lower demand',
    'mixed':   'non-monotonic — it rises and falls across the range',
    'flat':    'negligible',
}
_SHAPE = {
    'monotonic':     'monotonic',
    'non_monotonic': 'non-monotonic',
    'categorical':   'category-dependent',
    'near_flat':     'roughly flat',
    'flat':          'roughly flat',
}


def _importance_phrase(rank: int, n: int) -> str:
    if rank <= 3:
        return 'one of the strongest drivers of demand'
    if rank <= (2 * n) // 3:
        return 'a moderate driver of demand'
    return 'a minor driver of demand'


def generate_feature_template(xai: str, feature: str) -> str:
    """Deterministic per-feature description: [EFFECT] / [IMPORTANCE] / [RECOMMENDATION]."""
    d = describe_curve(xai, feature, explanations_dir=EXPLANATIONS_DIR)
    g = json.loads((EXPLANATIONS_DIR / f'global_{xai}_{LOSS_KEY}.json').read_text())
    imp = {it['feature']: it for it in g['global_importance']}[feature]
    n = len(g['global_importance'])
    fname = FEATURE_NAMES.get(feature, feature)
    has_peak = d['shape'] not in ('near_flat', 'flat')

    effect = (
        f"[EFFECT] The effect of {fname} on hourly demand is {_DIRECTION[d['direction']]}. "
        f"Its shape is {_SHAPE[d['shape']]}"
        + (f", with the contribution highest around {d['peak_label']}." if has_peak else '.')
    )
    importance = (
        f"[IMPORTANCE] {fname[0].upper() + fname[1:]} ranks {imp['rank']} of {n} features "
        f"by global importance (importance {imp['importance']:.3f}) — "
        f"{_importance_phrase(imp['rank'], n)}."
    )
    if has_peak:
        recommendation = (
            f"[RECOMMENDATION] Account for {fname} when planning bikes and staff: demand is "
            f"highest around {d['peak_label']}, so allocate more there and less at the opposite end."
        )
    else:
        recommendation = (
            f"[RECOMMENDATION] {fname[0].upper() + fname[1:]} barely moves demand; planning can "
            f"safely focus on the stronger drivers instead."
        )
    return '\n\n'.join([effect, importance, recommendation])


# Smoke test
print(generate_feature_template('ebm', 'temp'))
print('---')
print(generate_feature_template('ebm', 'weekday'))

[EFFECT] The effect of temperature on hourly demand is non-monotonic — it rises and falls across the range. Its shape is non-monotonic, with the contribution highest around ~31.2 C.

[IMPORTANCE] Temperature ranks 2 of 9 features by global importance (importance 0.253) — one of the strongest drivers of demand.

[RECOMMENDATION] Account for temperature when planning bikes and staff: demand is highest around ~31.2 C, so allocate more there and less at the opposite end.
---
[EFFECT] The effect of weekday on hourly demand is non-monotonic — it rises and falls across the range. Its shape is category-dependent, with the contribution highest around Friday.

[IMPORTANCE] Weekday ranks 7 of 9 features by global importance (importance 0.024) — a minor driver of demand.

[RECOMMENDATION] Account for weekday when planning bikes and staff: demand is highest around Friday, so allocate more there and less at the opposite end.


In [6]:
# Run the per-feature baseline for both models. Deterministic, resumable, no API, $0.
def _gen_feature_template(xai, feature, gen_idx):
    t0 = time.perf_counter()
    text = generate_feature_template(xai, feature)
    return build_global_record(
        form='template', model_name=xai, feature=feature, explanation=text,
        usage={'input_tokens': 0, 'output_tokens': 0, 'cache_read_input_tokens': 0},
        llm_model='template', elapsed_s=round(time.perf_counter() - t0, 6),
        extra={'scope': 'global', 'n_tool_calls': 0},
    )


FEATURES = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)
records = run_resumable_global_generation(
    form='template', model_names=XAI_MODELS, features=FEATURES,
    out_dir=GLOBAL_DIR, generate=_gen_feature_template,
)
print(f'\nDone. {len(records)} per-feature baselines saved in {GLOBAL_DIR} (no API, $0).')


Done. 18 per-feature baselines saved in /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/global (no API, $0).
